# Milestone 1 Cleaning Script

In [5]:
import pandas as pd

## Clean unanchored Events / timestamp column

In [6]:
df = pd.read_csv("../data/unanchored_events.csv.gz", sep=";")

df.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# identify different time formats
s = df["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 962480
European dd.mm.yyyy 239373
European d.mm.yyyy 239373
US-like mm/dd/yyyy 0


/var/folders/sn/nh99ycnd55q_9cvc_vgczkfw0000gn/T/ipykernel_32524/4130292105.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


invalid / custom strings 0
UTC 193657


In [8]:
def clean_unanchored_events_mixed_formats(
    df,
    timestamp_column="time:timestamp"
):
    df = df.copy()

    timestamps = df[timestamp_column].astype("string")

    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")

    # Format 1: German format, e.g. 31.12.2017 14:30:00
    mask_german = timestamps.str.match(
        r"^\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}:\d{2}$",
        na=False
    )

    parsed.loc[mask_german] = pd.to_datetime(
        timestamps.loc[mask_german],
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce",
        utc=True
    )

    # Format 2: ISO with T and Z, e.g. 2017-12-31T14:30:00.123456Z
    mask_iso_z = timestamps.str.match(
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$",
        na=False
    )

    parsed.loc[mask_iso_z] = pd.to_datetime(
        timestamps.loc[mask_iso_z],
        format="%Y-%m-%dT%H:%M:%S.%fZ",
        errors="coerce",
        utc=True
    )

    # Fallback: try normal pandas parsing for unchanged/default timestamps
    mask_remaining = parsed.isna() & timestamps.notna()

    parsed.loc[mask_remaining] = pd.to_datetime(
        timestamps.loc[mask_remaining],
        errors="coerce",
        utc=True
    )

    df[timestamp_column] = parsed

    return df

In [9]:
df_cleaned = clean_unanchored_events_mixed_formats(
    df,
    timestamp_column="time:timestamp"
)

In [10]:
s = df_cleaned["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 1201090
European dd.mm.yyyy 0
European d.mm.yyyy 0
US-like mm/dd/yyyy 0


/var/folders/sn/nh99ycnd55q_9cvc_vgczkfw0000gn/T/ipykernel_32524/4011362491.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


invalid / custom strings 0
UTC 0


In [11]:
# check for null values in timestamp column
df_check = df_cleaned.copy()

print(
    df_check.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application    222
Offer          192
Workflow       763
Name: time:timestamp, dtype: int64


In [12]:
application_na_rows = df_check[
    (df_check["EventOrigin"] == "Application") &
    (df_check["time:timestamp"].isna())
]

In [13]:
application_na_rows.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
16251,statechange,User_112,A_Validating,Application,ApplState_1428953723,complete,NaT,Car,New credit,Application_1462703151,5000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31740,statechange,User_27,A_Complete,Application,ApplState_1056054725,complete,NaT,Home improvement,Limit raise,Application_953097922,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48451,statechange,User_17,A_Complete,Application,ApplState_1591021281,complete,NaT,Car,New credit,Application_1979616665,7500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
67302,statechange,User_27,A_Accepted,Application,ApplState_1153503683,complete,NaT,Home improvement,Limit raise,Application_2122367472,30000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70429,statechange,User_18,A_Complete,Application,ApplState_768542349,complete,NaT,Existing loan takeover,New credit,Application_1901466720,50000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# check whole trace of one event with missing timestamp
case_id = application_na_rows.iloc[1]["case:concept:name"]

df_check.loc[
    df_check["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "EventID"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,EventID
31727,Application_953097922,A_Create Application,Application,2016-01-12 11:49:40+00:00,Application_953097922
31728,Application_953097922,A_Concept,Application,2016-01-12 11:49:40+00:00,ApplState_339072701
31729,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.257000+00:00,Workitem_190087837
31730,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.260000+00:00,Workitem_640792230
31731,Application_953097922,A_Accepted,Application,2016-01-12 11:50:34+00:00,ApplState_12271477
31732,Application_953097922,W_Complete application,Workflow,2016-01-12 11:50:56.698000+00:00,Workitem_1222575764
31733,Application_953097922,W_Complete application,Workflow,2016-01-12 13:32:08.186000+00:00,Workitem_545066525
31734,Application_953097922,O_Create Offer,Offer,2016-01-12 13:36:55.669000+00:00,Offer_1441589464
31735,Application_953097922,O_Created,Offer,2016-01-12 13:36:56.900000+00:00,OfferState_1404215056
31736,Application_953097922,O_Sent (mail and online),Offer,2016-01-12 13:37:10.953000+00:00,OfferState_1593764794


In [15]:
df_check["event_order"] = range(len(df_check))

In [16]:
def impute_missing_timestamps_within_cases(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()
    imputable_mask = missing_mask & previous_time.notna() & next_time.notna()

    df.loc[imputable_mask, timestamp_column] = (
        previous_time[imputable_mask]
        + (next_time[imputable_mask] - previous_time[imputable_mask]) / 2
    )

    df.loc[imputable_mask, "timestamp_cleaning"] = "imputed_between_neighbors"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [17]:
df_cleaned = impute_missing_timestamps_within_cases(
    df_check,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column="event_order"
)

In [18]:
df_cleaned.loc[
    df_cleaned["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "timestamp_cleaning"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,timestamp_cleaning
31727,Application_953097922,A_Create Application,Application,2016-01-12 11:49:40+00:00,NaN
31728,Application_953097922,A_Concept,Application,2016-01-12 11:49:40+00:00,NaN
31729,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.257000+00:00,NaN
31730,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.260000+00:00,NaN
31731,Application_953097922,A_Accepted,Application,2016-01-12 11:50:34+00:00,NaN
31732,Application_953097922,W_Complete application,Workflow,2016-01-12 11:50:56.698000+00:00,NaN
31733,Application_953097922,W_Complete application,Workflow,2016-01-12 13:32:08.186000+00:00,NaN
31734,Application_953097922,O_Create Offer,Offer,2016-01-12 13:36:55.669000+00:00,NaN
31735,Application_953097922,O_Created,Offer,2016-01-12 13:36:56.900000+00:00,NaN
31736,Application_953097922,O_Sent (mail and online),Offer,2016-01-12 13:37:10.953000+00:00,NaN


In [19]:
# check for null values in timestamp column
df_check = df_cleaned.copy()

print(
    df_check.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application    28
Offer           3
Workflow       31
Name: time:timestamp, dtype: int64


In [20]:
def fill_missing_timestamps_with_neighbor(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None,
    cleaning_column="timestamp_cleaning"
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()

    previous_mask = missing_mask & previous_time.notna()
    df.loc[previous_mask, timestamp_column] = previous_time[previous_mask]
    df.loc[previous_mask, cleaning_column] = "filled_from_previous_timestamp"

    missing_mask = df[timestamp_column].isna()

    next_mask = missing_mask & next_time.notna()
    df.loc[next_mask, timestamp_column] = next_time[next_mask]
    df.loc[next_mask, cleaning_column] = "filled_from_next_timestamp"

    missing_mask = df[timestamp_column].isna()
    df.loc[missing_mask, cleaning_column] = "could_not_fill_timestamp"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [21]:
df_cleaned = fill_missing_timestamps_with_neighbor(
    df_cleaned,
    case_column="case:concept:name",
    timestamp_column="time:timestamp"
)

In [22]:
df_cleaned["timestamp_cleaning"].value_counts()

timestamp_cleaning
imputed_between_neighbors         1115
filled_from_previous_timestamp      34
filled_from_next_timestamp          28
Name: count, dtype: int64

In [23]:
#export cleaned unanchored events to csv
df_cleaned.to_csv("../data/cleaned_unanchored_events.csv.gz", index=False, sep=";", compression="gzip")

## Cleaning Other Patterns

In [24]:
import pm4py

In [25]:
log_noised = pm4py.read_xes("../data/noised.xes.gz")
df_noised = pm4py.convert_to_dataframe(log_noised)

/Users/felixhauptmann/ProM-Assignment-Group-C/.venv/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

In [26]:
df_merged = df_noised.merge(
    df_cleaned[["EventID", "time:timestamp"]],
    on="EventID",
    how="left",
    suffixes=("_noised", "_clean")
)

df_merged["time:timestamp"] = df_merged["time:timestamp_clean"]

df_merged = df_merged.drop(columns=["time:timestamp_noised", "time:timestamp_clean"])

df_merged.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,...,CreditScore,OfferedAmount,OfferID,start_timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,pollution_type,time:timestamp
0,Created,User_1,A_Create Application,Application,Application_1000086665,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:57:21+00:00
1,statechange,User_1,A_Submitted,Application,ApplState_161925113,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:57:21+00:00
2,Created,User_1,W_Handle leads,Workflow,Workitem_747707399,schedule,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:57:21.963000+00:00
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1030261128,withdraw,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:58:28.286000+00:00
4,Created,User_1,W_Complete application,Workflow,Workitem_1127124826,schedule,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:58:28.293000+00:00


### Clean Polluted Labels

In [27]:
df_merged["concept:name"].value_counts()

concept:name
W_Call after offers                                            181250
W_Call incomplete files                                        111901
W_Complete application                                          98934
W_Handle leads                                                  44830
O_Create Offer                                                  40792
                                                                ...  
W_Complete application - Incident No. Application_988527938         1
W_Call after offers - Incident No. Application_989678604            1
W_Call after offers - Incident No. Application_992509096            1
W_Complete application - Incident No. Application_995859694         1
W_Complete application - Incident No. Application_999507989         1
Name: count, Length: 1014, dtype: int64

In [28]:
def clean_polluted_labels(
    df,
    activity_column="concept:name",
    cleaning_column="label_cleaning"
):
    df = df.copy()

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    original_labels = df[activity_column].copy()

    # convert as string
    df[activity_column] = df[activity_column].astype("string")

    patterns = [
        r"\._\d+$",                              # ._1691306052
        r"_\d+$",                                # _1691306052
        r"\s*-\s*Incident No\.?\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Incident\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Case\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Application\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*User\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Resource\s*[A-Za-z0-9_\-]+",
        r"\s*#\s*[A-Za-z0-9_\-]+",
        r"\s*\(\s*id\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
        r"\s*\(\s*case\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
    ]

    for pattern in patterns:
        df[activity_column] = df[activity_column].str.replace(
            pattern,
            "",
            regex=True
        )

    # normalize whitespace
    df[activity_column] = (
        df[activity_column]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    changed_mask = original_labels.astype("string") != df[activity_column]

    df.loc[changed_mask, cleaning_column] = "cleaned_polluted_label"

    return df

In [29]:
df_cleaned_final = df_merged.copy()

df_cleaned_final = clean_polluted_labels(
    df_cleaned_final,
    activity_column="concept:name"
)

In [30]:
df_cleaned_final["concept:name"].value_counts()

concept:name
W_Call after offers                           181447
W_Call incomplete files                       112064
W_Complete application                         99085
W_Handle leads                                 44878
O_Create Offer                                 40835
O_Created                                      40802
O_Sent (mail and online)                       37718
A_Create Application                           29978
A_Concept                                      29973
A_Accepted                                     29878
A_Complete                                     29836
W_Follow up files                              24057
W_Chase incomplete files                       23900
O_Returned                                     22169
A_Incomplete                                   21908
O_Cancelled                                    19875
A_Submitted                                    19444
O_Accepted                                     16344
A_Pending                        

### Clean Distorted Labels

In [31]:
DISTORTED_LABEL_MAPPING = {
    "W": {
        "Validate appl.": "Validate application",
        "Call after ofrs.": "Call after offers",
        "Call incompl. docs.": "Call incomplete files",
        "Call incompl. files": "Call incomplete files",
        "Complete appl.": "Complete application",
        "Handle lds.": "Handle leads",
        "Assess pot. frd.": "Assess potential fraud",
        "Assess pot. fraud": "Assess potential fraud",
        "Short. compl.": "Shortened completion",
        "Short. completion": "Shortened completion",
        "Pers. Ln. coll.": "Personal Loan collection",
        "Pers. Loan collection": "Personal Loan collection",
    },
    "O": {
        "Create Off.": "Create Offer",
        "Crtd.": "Created",
        "Sent (email and onl.)": "Sent (mail and online)",
        "Sent (mail and onl.)": "Sent (mail and online)",
        "Sent (onl. only)": "Sent (online only)",
        "Ret.": "Returned",
        "Canc.": "Cancelled",
        "Acc.": "Accepted",
        "Ref.": "Refused",
    },
    "A": {
        "Valid.": "Validating",
        "Create App.": "Create Application",
        "Conc.": "Concept",
        "Concept.": "Concept",
        "Acc.": "Accepted",
        "Comp.": "Complete",
        "Incompl.": "Incomplete",
        "Subm.": "Submitted",
        "Pend.": "Pending",
        "Canc.": "Cancelled",
        "Den.": "Denied",
    }
}

In [32]:
def build_label_mapping(grouped_mapping):
    mapping = {}

    for prefix, labels in grouped_mapping.items():
        for distorted, cleaned in labels.items():
            mapping[f"{prefix}_{distorted}"] = f"{prefix}_{cleaned}"

    return mapping

In [33]:
label_mapping = build_label_mapping(DISTORTED_LABEL_MAPPING)

In [34]:
label_mapping

{'W_Validate appl.': 'W_Validate application',
 'W_Call after ofrs.': 'W_Call after offers',
 'W_Call incompl. docs.': 'W_Call incomplete files',
 'W_Call incompl. files': 'W_Call incomplete files',
 'W_Complete appl.': 'W_Complete application',
 'W_Handle lds.': 'W_Handle leads',
 'W_Assess pot. frd.': 'W_Assess potential fraud',
 'W_Assess pot. fraud': 'W_Assess potential fraud',
 'W_Short. compl.': 'W_Shortened completion',
 'W_Short. completion': 'W_Shortened completion',
 'W_Pers. Ln. coll.': 'W_Personal Loan collection',
 'W_Pers. Loan collection': 'W_Personal Loan collection',
 'O_Create Off.': 'O_Create Offer',
 'O_Crtd.': 'O_Created',
 'O_Sent (email and onl.)': 'O_Sent (mail and online)',
 'O_Sent (mail and onl.)': 'O_Sent (mail and online)',
 'O_Sent (onl. only)': 'O_Sent (online only)',
 'O_Ret.': 'O_Returned',
 'O_Canc.': 'O_Cancelled',
 'O_Acc.': 'O_Accepted',
 'O_Ref.': 'O_Refused',
 'A_Valid.': 'A_Validating',
 'A_Create App.': 'A_Create Application',
 'A_Conc.': 'A_Con

In [35]:
def clean_distorted_labels(
    df,
    activity_column="concept:name"
):
    df = df.copy()

    label_mapping = build_label_mapping(DISTORTED_LABEL_MAPPING)

    df[activity_column] = df[activity_column].replace(label_mapping)

    return df

In [36]:
df_cleaned_final = clean_distorted_labels(df_cleaned_final)

In [37]:
df_cleaned_final["concept:name"].value_counts()

concept:name
W_Call after offers                           191092
W_Call incomplete files                       120572
W_Complete application                        106500
W_Handle leads                                 47264
O_Create Offer                                 42995
O_Created                                      42995
O_Sent (mail and online)                       39707
A_Create Application                           31509
A_Concept                                      31509
A_Accepted                                     31509
A_Complete                                     31362
W_Follow up files                              24057
W_Chase incomplete files                       23900
O_Returned                                     23305
A_Incomplete                                   23055
O_Cancelled                                    20898
A_Submitted                                    20423
O_Accepted                                     17228
A_Pending                        

### Clean Scattered Cases

In [38]:
# read CSV containing the scattered rows
validation_df = pd.read_csv("../data/scattered_events.csv.gz", sep=";")

In [39]:
# reindex
validation_df = validation_df.reindex(columns=df_cleaned_final.columns)

# merging the two dataframes
merged_df = pd.concat([df_cleaned_final, validation_df], ignore_index=True)

# parse timestamps
merged_df["time:timestamp"] = pd.to_datetime(
    merged_df["time:timestamp"],
    errors="coerce",
    utc=True
)

## Insert the missing cleaning scripts here!!!
After that we need an export to .xes.gz below (similar to the one in `polluter_script.ipynb`